# 레슨 04 — 데이터 불러오기와 저장 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. 이번 강의의 핵심은 CSV, Excel, JSON을 각각 읽는 법보다 여러 파일을 하나의 분석 산출물로 연결하는 흐름이다. 각 정답에는 코드와 함께 `왜 이 코드가 정답인지` 설명을 포함한다.

## 환경 셀

In [ ]:
import os
import pandas as pd
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/04/data"
else:
    DATA_BASE = "./data"
OUTPUT_PATH = "./quarter_summary.csv"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 1분기 CSV 첫 점검

In [ ]:
q1 = pd.read_csv(f"{DATA_BASE}/q1_sales.csv")

print("shape:", q1.shape)
print("columns:", list(q1.columns))
print("dtypes:")
print(q1.dtypes)
print("앞 5행:")
print(q1.head())
print("뒤 3행:")
print(q1.tail(3))
print("날짜 범위:", q1["date"].min(), "~", q1["date"].max())
print("1분기 매출 합계:", f"{q1['amount'].sum():,}원")
print("1분기 판매수량 합계:", f"{q1['units'].sum():,}개")

### 왜 이 코드가 정답인지

파일을 여러 개 다루기 전에 하나의 파일 구조를 먼저 확인해야 한다. `shape` 로 행과 열 개수를 보고, `columns` 로 열 이름을 확인하고, `dtypes` 로 계산 가능한 열을 구분한다. `head()` 와 `tail()` 은 앞뒤 행이 같은 구조인지 빠르게 확인하게 해 준다. 날짜 범위와 합계를 출력하면 이 파일이 정말 1분기 매출 파일인지 검증할 수 있다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 행 수 | 220 |
| 열 수 | 5 |
| 1분기 매출 합계 | 12,615,100원 |
| 1분기 판매수량 | 497개 |

---

## 문제 2 정답 — 네 개 CSV를 같은 방식으로 읽기

In [ ]:
frames = []

for q in range(1, 5):
    path = f"{DATA_BASE}/q{q}_sales.csv"
    temp = pd.read_csv(path)
    temp["quarter"] = q
    frames.append(temp)

    print(f"q{q} shape:", temp.shape)
    print(f"q{q} columns:", list(temp.columns))
    print(f"q{q} amount sum:", f"{temp['amount'].sum():,}원")

print("읽은 파일 수:", len(frames))

### 왜 이 코드가 정답인지

분기별 파일은 이름만 다르고 구조가 같다. 반복문으로 파일명을 만들면 네 파일을 같은 절차로 읽을 수 있어 실수를 줄인다. 각 DataFrame에 `quarter` 열을 추가해야 합친 뒤에도 어느 분기 데이터였는지 알 수 있다. `frames` 리스트는 여러 DataFrame을 모아 `concat` 할 준비 단계다.

**채점 포인트**

- 파일명을 하드코딩 네 줄로 반복하지 않아도 되지만, 네 파일을 모두 읽어야 한다.
- `quarter` 열을 합치기 전에 추가했는지 확인한다.
- 각 파일의 구조나 매출 합계를 최소 한 번 출력해야 한다.

---

## 문제 3 정답 — 분기별 CSV 합치기

In [ ]:
sales = pd.concat(frames, ignore_index=True)

print("sales shape:", sales.shape)
print("분기별 행 수:")
print(sales["quarter"].value_counts().sort_index())
print("전체 매출 합계:", f"{sales['amount'].sum():,}원")
print("전체 판매수량 합계:", f"{sales['units'].sum():,}개")
print(sales.head())

### 왜 이 코드가 정답인지

`pd.concat` 은 같은 구조의 표를 세로로 이어 붙일 때 사용한다. 분기별 파일이 각각 220행이므로 결과는 880행이어야 한다. `ignore_index=True` 를 사용하면 원래 파일마다 0부터 시작하던 인덱스를 하나의 연속 인덱스로 다시 만든다. `quarter` 별 행 수를 확인하면 네 파일이 빠짐없이 들어갔는지 검증할 수 있다.

**예상 핵심값**

```text
sales shape: (880, 6)
분기별 행 수: 1, 2, 3, 4 모두 220행
전체 매출 합계: 48,962,800원
```

---

## 문제 4 정답 — 날짜형 변환과 월 컬럼 만들기

In [ ]:
sales["date"] = pd.to_datetime(sales["date"])
sales["month"] = sales["date"].dt.to_period("M")

print("date dtype:", sales["date"].dtype)
print("날짜 범위:", sales["date"].min(), "~", sales["date"].max())

print("월별 행 수:")
print(sales.groupby("month")["sku"].count())

print("월별 매출 합계:")
print(sales.groupby("month")["amount"].sum())

### 왜 이 코드가 정답인지

CSV에서 읽은 날짜는 문자열이다. 월별 집계나 분기 추출을 안정적으로 하려면 `pd.to_datetime` 으로 날짜형으로 바꿔야 한다. `.dt.to_period("M")` 은 날짜를 월 단위 값으로 바꾼다. 이후 `groupby("month")` 로 월별 행 수와 매출 합계를 쉽게 계산할 수 있다.

**지도 메모**

문자열 상태에서도 일부 정렬이 되는 것처럼 보일 수 있다. 하지만 날짜형으로 바꾸지 않으면 요일, 월, 분기, 기간 차이 계산으로 확장하기 어렵다. 날짜형 변환은 4강 이후 데이터 분석에서 계속 반복되는 기본 습관이다.

---

## 문제 5 정답 — Excel 상품 기준표 읽기

In [ ]:
master = pd.read_excel(f"{DATA_BASE}/product_master.xlsx")

print("master shape:", master.shape)
print("columns:", list(master.columns))
print("dtypes:")
print(master.dtypes)
print(master)

print("sku 중복 개수:", master["sku"].duplicated().sum())
print("category 분포:")
print(master["category"].value_counts())
print("line 분포:")
print(master["line"].value_counts())

### 왜 이 코드가 정답인지

상품 기준표는 매출 파일에 없는 카테고리와 원가 정보를 제공한다. `pd.read_excel` 로 Excel 파일을 읽고, `sku` 가 중복되지 않는지 확인해야 안전하게 병합할 수 있다. 기준표의 `sku` 가 중복되면 매출 행이 병합 과정에서 늘어날 수 있다. `category` 와 `line` 분포를 보면 기준표가 어떤 상품군으로 구성되어 있는지 파악할 수 있다.

**채점 기준**

| 항목 | 통과 기준 |
|---|---|
| Excel 로드 | `product_master.xlsx` 를 `pd.read_excel` 로 읽음 |
| 키 확인 | `sku` 중복이 0개인지 확인 |
| 기준표 이해 | `category`, `unit_cost`, `line` 의 의미를 말할 수 있음 |

---

## 문제 6 정답 — 매출과 상품 기준표 병합

In [ ]:
merged = sales.merge(master, on="sku", how="left")

print("병합 전 행 수:", len(sales))
print("병합 후 행 수:", len(merged))
print("추가된 컬럼:", [col for col in merged.columns if col not in sales.columns])
print("상품 정보 결측:")
print(merged[["category", "unit_cost", "line"]].isna().sum())
print(merged.head())

print("모든 SKU 매칭 성공:", merged["category"].isna().sum() == 0)

### 왜 이 코드가 정답인지

매출 파일과 상품 기준표는 공통 열 `sku` 로 연결된다. `how="left"` 를 사용하면 매출 행을 모두 유지하면서 기준표 정보를 붙인다. 병합 후 행 수가 880행으로 유지되어야 하며, `category`, `unit_cost`, `line` 에 결측이 없어야 모든 SKU가 기준표와 매칭된 것이다. 병합은 실행만큼 검증이 중요하다.

**자주 틀리는 답안**

| 답안 | 문제 |
|---|---|
| `how="inner"` 사용 | 매칭 안 되는 SKU가 있을 때 매출 행이 사라질 수 있음 |
| 행 수 확인 생략 | 병합 오류를 놓치기 쉬움 |
| `sku` 대신 다른 열로 병합 | 기준표가 잘못 붙음 |

---

## 문제 7 정답 — 매출총이익과 이익률 계산

In [ ]:
merged["cost"] = merged["units"] * merged["unit_cost"]
merged["gross_profit"] = merged["amount"] - merged["cost"]
merged["profit_margin"] = merged["gross_profit"] / merged["amount"]

print(merged[["sku", "units", "amount", "unit_cost", "cost", "gross_profit", "profit_margin"]].head())

total_amount = merged["amount"].sum()
total_profit = merged["gross_profit"].sum()
overall_margin = total_profit / total_amount

print("전체 매출:", f"{total_amount:,.0f}원")
print("전체 매출총이익:", f"{total_profit:,.0f}원")
print("전체 이익률:", f"{overall_margin:.2%}")
print("행별 평균 이익률:", f"{merged['profit_margin'].mean():.2%}")

### 왜 이 코드가 정답인지

`amount` 는 판매금액이고 `unit_cost` 는 상품 1개당 원가다. 총원가는 `units * unit_cost` 로 계산해야 한다. 매출총이익은 판매금액에서 총원가를 뺀 값이다. 전체 이익률은 전체 매출총이익을 전체 매출로 나눈 값이다. 행별 이익률 평균과 전체 이익률은 의미가 다르므로 보고서에서는 전체 합계 기준 이익률을 우선 사용한다.

**손계산 예시**

수량 3개, 원가 10,000원, 판매금액 45,000원이면 총원가는 30,000원이고 매출총이익은 15,000원이다. 이익률은 15,000 / 45,000 = 33.3%다.

---

## 문제 8 정답 — 분기별 매출 요약

In [ ]:
quarter_summary = merged.groupby("quarter", as_index=False).agg(
    rows=("sku", "size"),
    amount=("amount", "sum"),
    units=("units", "sum"),
    gross_profit=("gross_profit", "sum"),
    avg_margin=("profit_margin", "mean"),
)

quarter_summary_sorted = quarter_summary.sort_values("amount", ascending=False)
print(quarter_summary_sorted)

best_amount_quarter = quarter_summary.loc[quarter_summary["amount"].idxmax(), "quarter"]
best_profit_quarter = quarter_summary.loc[quarter_summary["gross_profit"].idxmax(), "quarter"]

print("매출 1위 분기:", best_amount_quarter)
print("매출총이익 1위 분기:", best_profit_quarter)

### 왜 이 코드가 정답인지

분기 리포트는 `quarter` 별 집계가 핵심이다. `groupby("quarter").agg(...)` 로 행 수, 매출 합계, 판매수량 합계, 매출총이익 합계, 평균 이익률을 한 번에 계산한다. `as_index=False` 를 사용하면 `quarter` 가 인덱스가 아니라 일반 열로 남아 이후 병합이 편하다. 매출 1위와 이익 1위는 서로 다를 수 있으므로 각각 확인한다.

**예상 핵심값**

| 분기 | 매출 | 매출총이익 |
|---:|---:|---:|
| 1 | 12,615,100 | 4,685,300 |
| 2 | 11,732,400 | 4,475,200 |
| 3 | 11,140,300 | 4,315,500 |
| 4 | 13,475,000 | 5,236,200 |

---

## 문제 9 정답 — 카테고리와 상품 라인별 요약

In [ ]:
category_summary = merged.groupby("category").agg(
    amount=("amount", "sum"),
    units=("units", "sum"),
    gross_profit=("gross_profit", "sum"),
)
category_summary["profit_margin"] = category_summary["gross_profit"] / category_summary["amount"]
category_summary = category_summary.sort_values("amount", ascending=False)

line_summary = merged.groupby("line").agg(
    amount=("amount", "sum"),
    units=("units", "sum"),
    gross_profit=("gross_profit", "sum"),
)
line_summary["profit_margin"] = line_summary["gross_profit"] / line_summary["amount"]
line_summary = line_summary.sort_values("amount", ascending=False)

print("카테고리 요약:")
print(category_summary)
print("라인 요약:")
print(line_summary)

print("매출 1위 카테고리:", category_summary["amount"].idxmax())
print("이익 1위 카테고리:", category_summary["gross_profit"].idxmax())
print("basic 총이익:", f"{line_summary.loc['basic', 'gross_profit']:,.0f}원")
print("premium 총이익:", f"{line_summary.loc['premium', 'gross_profit']:,.0f}원")

### 왜 이 코드가 정답인지

상품 기준표를 병합했기 때문에 매출을 `category` 와 `line` 으로 묶을 수 있다. 카테고리별 매출과 이익은 상품군별 성과를 보여준다. 이익률은 단순 평균보다 `총이익 / 총매출` 로 계산해야 규모가 반영된다. basic과 premium 라인을 비교하면 상품 라인 단위의 성과도 볼 수 있다.

**예상 핵심값**

```text
매출 1위 카테고리: gadget
이익 1위 카테고리: gadget
basic 총이익: 16,092,800원
premium 총이익: 2,619,400원
```

---

## 문제 10 정답 — 매장별 실적 순위

In [ ]:
store_summary = merged.groupby("store_id").agg(
    amount=("amount", "sum"),
    units=("units", "sum"),
    gross_profit=("gross_profit", "sum"),
).sort_values("amount", ascending=False)

print("매장별 요약:")
print(store_summary)
print("상위 3개 매장:")
print(store_summary.head(3))

best_store_amount = store_summary["amount"].idxmax()
best_store_profit = store_summary["gross_profit"].idxmax()
store_amount_gap = store_summary["amount"].max() - store_summary["amount"].min()

print("매출 1위 매장:", best_store_amount)
print("이익 1위 매장:", best_store_profit)
print("매장 간 매출 최고-최저 차이:", f"{store_amount_gap:,.0f}원")

### 왜 이 코드가 정답인지

`store_id` 기준으로 묶으면 매장별 매출과 이익을 비교할 수 있다. 정렬 기준을 매출로 두면 매출 규모 순위가 나온다. 매출 1위와 이익 1위가 같은지 확인하면 매출 규모와 수익성이 함께 움직이는지 볼 수 있다. 최고-최저 차이는 매장 간 성과 격차를 숫자로 보여준다.

**예상 핵심값**

| 항목 | 값 |
|---|---|
| 매출 1위 매장 | W05 |
| 이익 1위 매장 | W05 |
| 매출 최고-최저 차이 | 2,793,200원 |

---

## 문제 11 정답 — SKU별 매출 상위 상품

In [ ]:
sku_summary = merged.groupby(["sku", "category", "line"]).agg(
    amount=("amount", "sum"),
    units=("units", "sum"),
    gross_profit=("gross_profit", "sum"),
)
sku_summary["avg_price"] = sku_summary["amount"] / sku_summary["units"]

top_sku_by_amount = sku_summary.sort_values("amount", ascending=False)
top_sku_by_units = sku_summary.sort_values("units", ascending=False)

print("매출 상위 5개 SKU:")
print(top_sku_by_amount.head(5))
print("수량 상위 5개 SKU:")
print(top_sku_by_units.head(5))

best_amount_sku = top_sku_by_amount.index[0][0]
best_units_sku = top_sku_by_units.index[0][0]

print("매출 1위 SKU:", best_amount_sku)
print("수량 1위 SKU:", best_units_sku)
print("매출 1위와 수량 1위가 같은가:", best_amount_sku == best_units_sku)

### 왜 이 코드가 정답인지

SKU별 분석은 가장 구체적인 상품 단위 성과를 보여준다. `sku`, `category`, `line` 을 함께 groupby 하면 상품 코드와 기준표 정보가 같이 남는다. 평균 판매단가는 `amount / units` 로 계산해야 전체 판매수량을 반영한다. 매출 1위와 수량 1위가 다를 수 있는데, 고가 상품은 수량이 적어도 매출이 클 수 있기 때문이다.

**예상 핵심값**

```text
매출 1위 SKU: GD02
수량 1위 SKU: TY01
매출 1위와 수량 1위가 같은가: False
```

---

## 문제 12 정답 — JSON 캠페인 로그 읽기

In [ ]:
events = pd.read_json(f"{DATA_BASE}/campaign_events.json")

print("events shape:", events.shape)
print(events.head())
print("dtypes:")
print(events.dtypes)

events["date"] = pd.to_datetime(events["date"])
events["month"] = events["date"].dt.to_period("M")
events["quarter"] = events["date"].dt.quarter
events["conversion_rate"] = events["conversions"] / events["clicks"]

print(events.head())
print("채널 목록:", events["channel"].unique())

### 왜 이 코드가 정답인지

JSON이 records 형태면 `pd.read_json` 으로 바로 표 형태로 읽을 수 있다. 캠페인 로그는 날짜, 채널, 캠페인명, 클릭 수, 전환 수를 가진다. 날짜형 변환 후 `month`, `quarter` 를 만들면 매출 데이터와 같은 시간 단위로 비교할 수 있다. 전환율은 전환 수를 클릭 수로 나눈 값이다.

**채점 포인트**

- JSON 파일을 DataFrame으로 읽었는가.
- 날짜형 변환 후 월/분기 열을 만들었는가.
- `conversion_rate` 를 계산했는가.

---

## 문제 13 정답 — 캠페인 채널별 성과 요약

In [ ]:
campaign_channel = events.groupby("channel").agg(
    clicks=("clicks", "sum"),
    conversions=("conversions", "sum"),
)
campaign_channel["conversion_rate"] = campaign_channel["conversions"] / campaign_channel["clicks"]
campaign_channel = campaign_channel.sort_values("conversions", ascending=False)

monthly_conversions = events.groupby("month")["conversions"].sum()

print("캠페인 채널별 성과:")
print(campaign_channel)
print("전환 수 1위 채널:", campaign_channel["conversions"].idxmax())
print("전환율 1위 채널:", campaign_channel["conversion_rate"].idxmax())
print("월별 전환 수:")
print(monthly_conversions)

### 왜 이 코드가 정답인지

캠페인 성과는 클릭과 전환을 함께 봐야 한다. 전환 수가 많은 채널은 규모가 큰 채널이고, 전환율이 높은 채널은 클릭 대비 효율이 좋은 채널이다. 전환율은 채널별 합계 기준으로 `전환 합계 / 클릭 합계` 를 계산해야 클릭 수 차이가 반영된다. 월별 전환 수는 특정 월에 캠페인 성과가 몰렸는지 확인하게 해 준다.

**예상 핵심값**

```text
전환 수 1위 채널: store
전환율 1위 채널: store
```

---

## 문제 14 정답 — 매출과 캠페인 분기 리포트 결합

In [ ]:
campaign_quarter = events.groupby("quarter", as_index=False).agg(
    clicks=("clicks", "sum"),
    conversions=("conversions", "sum"),
)

final_report = quarter_summary.merge(campaign_quarter, on="quarter", how="left")
final_report["profit_per_conversion"] = final_report["gross_profit"] / final_report["conversions"]

print("최종 분기 리포트:")
print(final_report)

final_report.to_csv(OUTPUT_PATH, index=False)
print("저장 경로:", OUTPUT_PATH)
print("파일 생성 확인:", os.path.exists(OUTPUT_PATH))

### 왜 이 코드가 정답인지

매출 데이터와 캠페인 데이터는 모두 `quarter` 기준으로 요약할 수 있다. 두 요약표를 `quarter` 로 병합하면 분기별 매출, 이익, 클릭, 전환을 한 표에서 볼 수 있다. `profit_per_conversion` 은 캠페인 전환 1건당 매출총이익을 보는 지표다. 마지막에 `to_csv` 로 저장하고 `os.path.exists` 로 파일 생성 여부를 확인하면 저장 실습까지 완료된다.

**예상 핵심값**

| 분기 | 매출총이익 | 전환 | 이익/전환 |
|---:|---:|---:|---:|
| 1 | 4,685,300 | 3,221 | 약 1,455 |
| 2 | 4,475,200 | 2,259 | 약 1,981 |
| 3 | 4,315,500 | 2,788 | 약 1,548 |
| 4 | 5,236,200 | 2,866 | 약 1,827 |

---

## 문제 15 정답 — 분기별 파일 통합 리포트 결론

In [ ]:
best_amount_quarter = final_report.loc[final_report["amount"].idxmax(), "quarter"]
best_profit_quarter = final_report.loc[final_report["gross_profit"].idxmax(), "quarter"]
best_conversion_quarter = final_report.loc[final_report["conversions"].idxmax(), "quarter"]
best_efficiency_quarter = final_report.loc[final_report["profit_per_conversion"].idxmax(), "quarter"]

print("매출 1위 분기:", best_amount_quarter)
print("매출총이익 1위 분기:", best_profit_quarter)
print("캠페인 전환 1위 분기:", best_conversion_quarter)
print("이익/전환 1위 분기:", best_efficiency_quarter)
print(final_report)

### 왜 이 코드가 정답인지

최종 결론은 `final_report` 에서 대표 지표를 정확히 꺼내는 문제다. 매출 1위, 매출총이익 1위, 전환 1위, 이익/전환 1위는 서로 다른 관점이다. 같은 분기가 여러 지표에서 1위일 수 있지만, 기준을 분리해 출력해야 어떤 관점에서 좋은지 명확해진다. 보고서는 숫자를 계산하는 데서 끝나지 않고, 어떤 행동을 할지 연결해야 한다.

**결론 예시**

```text
4분기는 매출과 매출총이익이 모두 가장 높아 연말 판매 성과가 가장 강했다.
캠페인 전환 수는 1분기가 가장 높았지만, 이익/전환 효율은 2분기가 가장 좋았다.
따라서 다음 분석에서는 4분기의 판매 상품 구성을 유지하되, 2분기의 캠페인 효율이 높았던 이유를 채널별로 확인하겠다.
최종 리포트는 quarter_summary.csv 로 저장했으므로 다른 도구에서 이어서 검토할 수 있다.
```

---

## 전체 채점 메모

| 구간 | 문제 | 핵심 개념 | 필수 통과 조건 |
|---|---|---|---|
| CSV 로드 | 1~4 | read_csv, concat, 날짜형 | 네 분기 CSV 880행 통합 |
| Excel 결합 | 5~7 | read_excel, merge, 계산 열 | SKU 기준 병합과 이익 계산 |
| 매출 요약 | 8~11 | groupby, 정렬 | 분기/카테고리/매장/SKU 요약 |
| JSON 처리 | 12~13 | read_json, 전환율 | 채널별 캠페인 성과 계산 |
| 저장/결론 | 14~15 | merge, to_csv, 리포트 | final_report 저장과 근거 기반 결론 |

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 교사 피드백 |
|---|---|---|
| 파일 하나만 읽고 끝냄 | 통합 리포트 목적 누락 | 4개 CSV를 모두 읽어야 분기 비교 가능 |
| `quarter` 열을 합친 뒤 만들려고 함 | 파일별 출처 정보 소실 | 합치기 전에 각 파일에 분기 번호 추가 |
| 병합 후 행 수 확인 생략 | merge 오류 감지 어려움 | 병합 전후 행 수와 결측을 반드시 출력 |
| `unit_cost` 만 빼서 이익 계산 | 수량 반영 누락 | 원가는 `units * unit_cost` |
| 전환율을 단순 평균으로만 사용 | 클릭 수 규모 반영 부족 | 보고서용 전환율은 합계 기준 |
| 저장 경로가 개인 절대경로 | 다른 환경에서 실패 | `OUTPUT_PATH` 또는 상대 경로 사용 |

## 부분 점수 운영 기준

1. 문제 1~3에서 CSV 통합이 안 되면 이후 분석의 기준 데이터가 없으므로 먼저 보충한다.
2. 문제 5~6에서 Excel을 못 읽는 경우 환경 문제인지 코드 문제인지 분리한다. `openpyxl` 설치 오류면 환경 설명 후 코드 자체는 부분 인정할 수 있다.
3. 문제 7의 이익 계산이 틀리면 문제 8~15의 이익 관련 결론은 모두 재검토한다.
4. 문제 12~13 캠페인 분석이 빠져도 매출 리포트는 부분 통과 가능하지만, 4강의 파일 형식 통합 목표에는 미달이다.
5. 결론에 저장 파일 언급이 없거나 숫자가 없으면 코드와 별도로 결론 셀을 다시 작성하게 한다.

## 문제별 지도 질문

| 문제 | 학생에게 던질 질문 | 확인할 답 |
|---:|---|---|
| 1 | CSV 하나를 읽은 뒤 가장 먼저 확인할 것은 무엇인가요? | shape, columns, head |
| 2 | 합치기 전에 `quarter` 열을 넣는 이유는 무엇인가요? | 출처 분기 보존 |
| 3 | `ignore_index=True` 는 왜 쓰나요? | 합친 뒤 인덱스를 새로 정리 |
| 4 | 날짜를 문자열로 둔 것과 날짜형으로 바꾼 것의 차이는 무엇인가요? | 월/분기/기간 계산 가능 |
| 5 | 기준표에서 `sku` 중복을 확인하는 이유는 무엇인가요? | 병합 시 행 중복 증가 방지 |
| 6 | left join 후 결측이 생기면 무엇을 의심해야 하나요? | 매출 SKU가 기준표에 없음 |
| 7 | 매출총이익 공식은 무엇인가요? | amount - units * unit_cost |
| 8 | 매출 1위와 이익 1위는 왜 따로 봐야 하나요? | 규모와 수익성이 다를 수 있음 |
| 9 | 카테고리 이익률은 어떻게 계산했나요? | 총이익 / 총매출 |
| 10 | 매장 간 격차는 어떤 값으로 봤나요? | 매출 최고-최저 차이 |
| 11 | SKU 평균 판매단가는 어떻게 계산하나요? | SKU 매출 / SKU 수량 |
| 12 | JSON 로그를 월/분기로 바꾸는 이유는 무엇인가요? | 매출 데이터와 시간 기준 맞춤 |
| 13 | 전환 수와 전환율은 어떻게 다른가요? | 규모 vs 효율 |
| 14 | 저장 파일 생성 여부는 어떻게 확인하나요? | `os.path.exists` |
| 15 | 결론에 들어갈 최소 요소는 무엇인가요? | 숫자, 비교 기준, 다음 행동 |

## 보충 설명 포인트

- 이번 강의의 중심은 파일 형식이 아니라 "분리된 파일을 하나의 리포트로 만드는 순서"다. 파일을 읽는 순간마다 구조 확인, 합친 뒤 행 수 확인, 병합 뒤 결측 확인을 반복하게 한다.
- Excel 기준표는 실무에서 마스터 데이터 역할을 한다. 학생이 "왜 굳이 Excel을 붙이나요?"라고 묻는다면, SKU만 있는 매출표에 카테고리와 원가라는 의미를 붙이기 위해서라고 설명한다.
- JSON은 API 응답이나 로그에서 자주 나온다. 4강에서는 깊은 JSON 정규화보다 records 형태를 DataFrame으로 읽어 같은 groupby 흐름에 올리는 경험이 중요하다.
- 저장 실습은 분석 결과를 다음 도구로 넘기는 단계다. 저장 파일이 원본 데이터가 아니라 요약 리포트인지 확인하게 한다.

## 재실행 확인 순서

1. 런타임을 새로 시작한다.
2. 환경 셀부터 문제 15까지 순서대로 실행한다.
3. `sales.shape` 가 `(880, 8)` 근처인지, `merged` 행 수가 880인지 확인한다.
4. `final_report` 에 분기 1~4가 모두 있는지 확인한다.
5. `quarter_summary.csv` 가 생성되었는지 확인하고, 필요하면 제출 전 삭제해도 된다.

정답 코드와 학생 코드가 다르더라도 같은 파일을 읽고, 같은 기준으로 합치고, 같은 핵심 지표를 계산하면 통과로 본다. 반대로 코드가 실행되어도 어떤 파일에서 어떤 기준으로 온 숫자인지 설명하지 못하면 보충 설명을 요구한다.